# Просмотр `.mat` — кадры по слайдеру

Выбери файл в списке и двигай **ползунок** (или колёсико на слайдере), чтобы листать кадры. Справа — маска из `datasets/datasets_list/*/masks/`.

Диапазон цветов по умолчанию — **percentile** по всему видео (как в `thermal_to_video.py`), чтобы кадры не сливались в один цвет.

Kernel: `/Users/user/Education/CVYandexCamp/venv/bin/python`

In [ ]:
DATA_DIR = None               # None → <repo>/data ; или абсолютный путь
PATTERN = "*.mat"             # R_*.mat | Z_*.mat | sample*.mat
SAMPLE_TIME_AXIS = 2          # только для sample*.mat (если авто не угадает)
CMAP = "inferno"              # colormap для тепловизора
CLIM_MODE = "percentile"      # percentile | global | frame
PERCENTILE = 1.0              # для percentile: диапазон [P, 100-P] по всему видео

In [ ]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from PIL import Image


def find_project_root() -> Path:
    """Корень репо: есть папки data/ и irt_data/. Не зависит от cwd."""
    start = Path.cwd().resolve()
    candidates = [start]
    if start.name == "notebooks":
        candidates.append(start.parent)
    candidates.append(start / "thermal-control-ya-project")
    seen: set[Path] = set()
    for base in candidates:
        for p in [base, *base.parents]:
            p = p.resolve()
            if p in seen:
                continue
            seen.add(p)
            if (p / "data").is_dir() and (p / "irt_data").is_dir():
                return p
    raise FileNotFoundError(
        "Не найден корень репозитория (нужны data/ и irt_data/). "
        f"cwd={start}. Укажи DATA_DIR абсолютным путём."
    )


def mask_search_dirs(root: Path) -> list[Path]:
    return [
        root / "datasets/datasets_list/dataset_kaggle/masks",
        root / "datasets/datasets_list/dataset_tpu/masks",
        root / "datasets/dataset_tpu/labels/table_mask",
    ]


def find_mask_path(mat_path: Path, root: Path) -> Path | None:
    stem = mat_path.stem
    for directory in mask_search_dirs(root):
        if not directory.is_dir():
            continue
        candidates = [directory / f"{stem}.png"]
        m = re.match(r"sample(\d+)$", stem, re.I)
        if m:
            n = int(m.group(1))
            candidates.append(directory / f"Sample_{n}_Static.png")
        for candidate in candidates:
            if candidate.is_file():
                return candidate
        target = stem.replace(" ", "_").lower()
        for png in directory.glob("*.png"):
            if png.stem.replace(" ", "_").lower() == target:
                return png
    return None


def load_mask(mat_path: Path, size: tuple[int, int], root: Path) -> tuple[np.ndarray | None, Path | None]:
    mask_path = find_mask_path(mat_path, root)
    if mask_path is None:
        return None, None
    arr = np.asarray(Image.open(mask_path))
    if arr.ndim == 3:
        arr = arr[..., 0]
    h, w = size
    if arr.shape != (h, w):
        arr = np.asarray(Image.fromarray(arr).resize((w, h), resample=Image.NEAREST))
    return arr, mask_path


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from irt_data.cache import _load_mat_array

if DATA_DIR is None:
    data_dir = ROOT / "data"
elif Path(DATA_DIR).is_absolute():
    data_dir = Path(DATA_DIR)
else:
    data_dir = (ROOT / DATA_DIR).resolve()

mat_files = sorted(data_dir.glob(PATTERN))
if not mat_files:
    raise FileNotFoundError(f"Нет файлов {PATTERN} в {data_dir}")

with_mask = sum(1 for p in mat_files if find_mask_path(p, ROOT))
print(f"ROOT={ROOT}")
print(f"Найдено {len(mat_files)} файлов в {data_dir} ({with_mask} с маской)")

In [ ]:
def _time_axis_for(path: Path) -> int | None:
    return SAMPLE_TIME_AXIS if path.name.lower().startswith("sample") else None


def load_video(path: Path) -> tuple[np.ndarray, dict]:
    video, meta = _load_mat_array(path, time_axis=_time_axis_for(path))
    return video, meta


def _clim_for_video(video: np.ndarray) -> tuple[float, float]:
    if CLIM_MODE == "frame":
        return float("nan"), float("nan")
    if CLIM_MODE == "percentile":
        lo, hi = np.percentile(video, [PERCENTILE, 100.0 - PERCENTILE])
    else:
        lo, hi = float(video.min()), float(video.max())
    if lo >= hi:
        pad = max(abs(lo) * 1e-6, 1e-3)
        lo, hi = lo - pad, hi + pad
    return float(lo), float(hi)


def _clim_for_frame(frame: np.ndarray, video_clim: tuple[float, float]) -> tuple[float, float]:
    if CLIM_MODE == "frame":
        lo, hi = float(frame.min()), float(frame.max())
    else:
        lo, hi = video_clim
    if lo >= hi:
        pad = max(abs(lo) * 1e-6, 1e-3)
        lo, hi = lo - pad, hi + pad
    return lo, hi


state: dict = {
    "video": None,
    "meta": None,
    "path": None,
    "clim": (0.0, 1.0),
    "mask": None,
    "mask_path": None,
}

fig, (ax_frame, ax_mask) = plt.subplots(1, 2, figsize=(11, 5))
fig.tight_layout()
im = ax_frame.imshow(np.zeros((2, 2)), cmap=CMAP, origin="upper")
cb = fig.colorbar(im, ax=ax_frame, fraction=0.046, pad=0.04)
title = fig.suptitle("")
ax_frame.set_title("кадр")
ax_frame.axis("off")
im_mask = ax_mask.imshow(np.zeros((2, 2)), cmap="gray", vmin=0, vmax=255, origin="upper")
ax_mask.set_title("маска")
ax_mask.axis("off")
plot_out = widgets.Output()


def _render_frame(t: int) -> None:
    video = state["video"]
    meta = state["meta"]
    path = state["path"]
    if video is None:
        return
    t = int(np.clip(t, 0, len(video) - 1))
    frame = video[t]
    vmin, vmax = _clim_for_frame(frame, state["clim"])
    im.set_data(frame)
    im.set_clim(vmin, vmax)
    cb.update_normal(im)

    mask = state["mask"]
    mask_path = state["mask_path"]
    if mask is not None:
        im_mask.set_data(mask)
        mask_label = mask_path.name if mask_path else "маска"
        defect_pct = 100.0 * float((mask > 0).mean())
        ax_mask.set_title(f"маска: {mask_label} ({defect_pct:.1f}% defect)")
    else:
        im_mask.set_data(np.zeros(frame.shape, dtype=np.uint8))
        ax_mask.set_title("маска: не найдена")

    fps = meta.get("fps")
    time_s = f" | t={t / fps:.2f}s" if fps else ""
    title.set_text(
        f"{path.name}  frame {t}/{len(video)-1}{time_s}\n"
        f"shape (T,H,W)={video.shape}  key={meta.get('key')}  "
        f"clim=[{vmin:.3g}, {vmax:.3g}] ({CLIM_MODE})"
    )
    with plot_out:
        plot_out.clear_output(wait=True)
        display(fig)


file_dd = widgets.Dropdown(
    options=[(p.name, p) for p in mat_files],
    description="файл:",
    layout=widgets.Layout(width="420px"),
)
frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=0,
    step=1,
    description="кадр:",
    continuous_update=True,
    readout=True,
    layout=widgets.Layout(width="520px"),
)
jump = widgets.BoundedIntText(
    value=0, min=0, max=0, description="перейти:", layout=widgets.Layout(width="180px")
)
btn_go = widgets.Button(description="→", layout=widgets.Layout(width="40px"))


def _set_file(path: Path) -> None:
    print(f"загрузка {path.name}…", flush=True)
    video, meta = load_video(path)
    mask, mask_path = load_mask(path, (video.shape[1], video.shape[2]), ROOT)
    state.update(
        video=video,
        meta=meta,
        path=path,
        clim=_clim_for_video(video),
        mask=mask,
        mask_path=mask_path,
    )
    last = len(video) - 1
    frame_slider.max = last
    jump.max = last
    frame_slider.value = 0
    jump.value = 0
    _render_frame(0)
    mask_info = mask_path.name if mask_path else "нет"
    print(
        f"готово: T={video.shape[0]} H={video.shape[1]} W={video.shape[2]}  "
        f"range=[{video.min():.3g}, {video.max():.3g}]  clim={state['clim']}  mask={mask_info}"
    )


def _on_file(change) -> None:
    if change["name"] == "value" and change["new"] is not None:
        _set_file(change["new"])


def _on_frame(change) -> None:
    if change["name"] == "value":
        jump.value = change["new"]
        _render_frame(change["new"])


def _on_go(_btn) -> None:
    frame_slider.value = int(jump.value)


file_dd.observe(_on_file, names="value")
frame_slider.observe(_on_frame, names="value")
btn_go.on_click(_on_go)

display(
    widgets.VBox([
        file_dd,
        widgets.HBox([frame_slider, jump, btn_go]),
        plot_out,
    ])
)

_set_file(mat_files[0])